# hscredit.core.eda 模块完整演示

本示例演示 hscredit EDA (Exploratory Data Analysis) 模块的完整功能，包括：

1. **数据概览与质量评估** - 数据信息、缺失分析、特征摘要
2. **目标变量分析** - 逾期分布、逾期率分析、趋势分析
3. **特征分析** - 数值/类别特征分布、异常值检测
4. **特征标签关系** - IV分析、WOE分析、分箱逾期率
5. **相关性分析** - 相关系数、高相关特征对、VIF分析
6. **稳定性分析** - PSI、CSI、时间稳定性追踪
7. **Vintage分析** - 账龄分析、滚动率分析
8. **综合报告** - 一键生成完整数据探索报告

**数据准备**：使用分箱器将评分分箱为客户等级 A~E 和 N（缺失）

## 1. 环境初始化与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 导入 hscredit
import hscredit
from hscredit import init_setting
from hscredit.core import eda
from hscredit.core.binning import OptimalBinning, QuantileBinning

# 初始化环境（设置中文字体等）
init_setting()

print(f"hscredit 版本: {hscredit.__version__}")

In [ ]:
# 加载示例数据
df = pd.read_excel('hscredit.xlsx')

print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n数据前5行:")
df.head()

数据形状: (12448, 85)

列名: ['MOB1', 'MOB2', 'loan_date', 'bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_confidence', 'consumer_finance_loan_credit_line', 'consumer_finance_loan_credit_line_avg', 'consumer_finance_loan_credit_line_max', 'consumer_finance_loan_lender_count', 'consumer_finance_loan_product_count', 'credit_loan_duration', 'hit_consumer_finance_lender_count', 'hit_lender_count', 'hit_network_loan_lender_count', 'last_performance_days', 'lender_count_12m', 'lender_count_1m', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_1m', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_coun

,MOB1,MOB2,loan_date,bj_qy24,mobtech80,bairong_v1,xiaoniu_c3,dxm_v6pro,bcf_fpv1,apply_for_admission_confidence,...,performance_loan_count_24m,performance_loan_count_3m,performance_loan_count_6m,performance_loan_sum_12m,performance_loan_sum_1m,performance_loan_sum_24m,performance_loan_sum_3m,performance_loan_sum_6m,FPD15,SFPD15
0,0,0,2025-08-01,612,718,914.7400,687,NaN,NaN,74,...,231,70,134,10,7,10,10,10,0,0
1,0,0,2025-04-14,581,709,822.1200,629,53.3800,306.0000,73,...,275,64,113,9,7,10,8,9,0,0
2,0,0,2025-08-27,621,718,814.6800,543,NaN,NaN,75,...,177,47,84,10,7,10,9,10,0,0
3,0,0,2025-08-18,594,698,707.1500,507,NaN,NaN,75,...,158,23,39,10,6,10,9,9,0,0
4,0,48,2025-07-23,563,718,777.3400,492,NaN,NaN,75,...,302,70,129,10,7,10,9,10,0,1


## 2. 数据预处理 - 创建客户等级（A~E + N）

使用分箱器将评分字段分箱为5个等级，并将缺失值标记为 'N'

In [ ]:
# 识别数值型评分相关字段
score_cols = [col for col in df.columns if 'score' in col.lower() or 'rating' in col.lower()]
print(f"评分相关字段: {score_cols}")

# 如果没有评分字段，选择几个数值特征作为示例
if not score_cols:
    # 选择数值型特征
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # 排除ID、目标变量等
    exclude_cols = ['target', 'label', 'id', 'fpd', 'dpd']
    score_cols = [col for col in numeric_cols[:5] if not any(e in col.lower() for e in exclude_cols)]
    print(f"选择的数值特征: {score_cols}")

评分相关字段: ['apply_for_admission_score', 'loan_behavior_score']


In [ ]:
# 使用分箱器创建客户等级
def create_customer_grade(df, score_col, grade_col_name='customer_grade'):
    """
    将评分分箱为 A~E 等级，缺失值为 N
    A: 最优, B: 良好, C: 一般, D: 较差, E: 最差, N: 缺失
    """
    data = df.copy()
    
    # 使用等频分箱（5箱）
    binner = QuantileBinning(max_n_bins=5, missing_separate=True)
    
    # 创建临时目标变量（用于分箱）
    if 'target' in data.columns:
        y = data['target']
    else:
        # 随机生成临时目标变量
        np.random.seed(42)
        y = pd.Series(np.random.randint(0, 2, len(data)))
    
    # 拟合分箱器
    valid_mask = data[score_col].notna()
    if valid_mask.sum() > 10:
        binner.fit(data[[score_col]], y)
        
        # 转换分箱
        bins = binner.transform(data[[score_col]], metric='indices')
        
        # 映射为等级标签
        grade_map = {0: 'E', 1: 'D', 2: 'C', 3: 'B', 4: 'A'}  # 分数越低等级越差
        data[grade_col_name] = bins[score_col].map(grade_map)
        
        # 缺失值标记为 N
        data[grade_col_name] = data[grade_col_name].fillna('N')
        
        print(f"{score_col} -> {grade_col_name}")
        print(f"等级分布:\n{data[grade_col_name].value_counts().sort_index()}")
    else:
        data[grade_col_name] = 'N'
    
    return data

# 选择第一个评分/数值字段创建客户等级
if score_cols:
    df = create_customer_grade(df, score_cols[0], 'customer_grade')
    df.head()

apply_for_admission_score -> customer_grade
等级分布:
customer_grade
A    2664
B    4250
C    1112
D    2244
E    2178
Name: count, dtype: int64


In [ ]:
# 为多个数值特征创建等级（用于后续分析）
if len(score_cols) >= 3:
    for i, col in enumerate(score_cols[:3]):
        df = create_customer_grade(df, col, f'grade_{i+1}')

# 查看新增字段
grade_cols = [col for col in df.columns if 'grade' in col]
print(f"\n客户等级字段: {grade_cols}")


客户等级字段: ['customer_grade']


## 3. 数据概览与质量评估

In [ ]:
### 3.1 数据基本信息
print("="*60)
print("3.1 数据基本信息 (data_info)")
print("="*60)

info_df = eda.data_info(df)
info_df

3.1 数据基本信息 (data_info)


,信息项,值
0,样本数（行）,12448.0000
1,特征数（列）,86.0000
2,数值型特征,83.0000
3,分类型特征,1.0000
4,日期型特征,1.0000
5,常数特征,1.0000
6,ID特征,0.0000
7,缺失值列数,2.0000
8,总缺失值数,20368.0000
9,内存使用(MB),8.7600


In [ ]:
### 3.2 缺失值分析
print("\n" + "="*60)
print("3.2 缺失值分析 (missing_analysis)")
print("="*60)

missing_df = eda.missing_analysis(df)
missing_df.head(20)


3.2 缺失值分析 (missing_analysis)


,特征名,缺失数,缺失率(%),非空数,查得率(%)
0,dxm_v6pro,10184,81.8100,2264,18.1900
1,bcf_fpv1,10184,81.8100,2264,18.1900
2,MOB1,0,0.0000,12448,100.0000
3,network_loan_confidence,0,0.0000,12448,100.0000
4,normal_repayment_order_in_total_order_rate,0,0.0000,12448,100.0000
5,network_loan_product_count,0,0.0000,12448,100.0000
6,network_loan_lender_count_24m,0,0.0000,12448,100.0000
7,network_loan_lender_count_12m,0,0.0000,12448,100.0000
8,network_loan_lender_count,0,0.0000,12448,100.0000
9,network_loan_credit_line_max,0,0.0000,12448,100.0000


In [ ]:
### 3.3 特征摘要
print("\n" + "="*60)
print("3.3 特征摘要 (feature_summary)")
print("="*60)

# 自动推断特征类型
summary_df = eda.feature_summary(df)
summary_df.head(20)


3.3 特征摘要 (feature_summary)


,特征名,字段类型,样本数,缺失数,缺失率,唯一值数,众数,众数频数,众数占比,零值数,...,平均值,标准差,1%,5%,25%,50%,75%,95%,99%,PSI
0,MOB1,numerical,12448,0,0.0000,95,0,10838,87.0700,10838,...,4.0794,15.4973,0.0000,0.0000,0.0000,0.0000,0.0000,48.0000,77.0000,0.0000
1,MOB2,numerical,12448,0,0.0000,105,0,9341,75.0400,9341,...,6.7825,17.6641,0.0000,0.0000,0.0000,0.0000,0.0000,52.0000,77.0000,0.0000
2,loan_date,datetime,12448,0,0.0000,223,2025-08-08 00:00:00,327,2.6300,0,...,NaN,NaN,2025-08-08 00:00:00,2025-08-09 00:00:00,2025-08-13 00:00:00,2025-08-23 00:00:00,2025-07-02 00:00:00,2025-04-17 00:00:00,2025-03-23 00:00:00,NaN
3,bj_qy24,numerical,12448,0,0.0000,349,599,113,0.9100,0,...,598.5537,67.1235,461.0000,489.0000,553.0000,599.0000,643.0000,707.0000,756.5300,0.0008
4,mobtech80,numerical,12448,0,0.0000,22,718,5839,46.9100,0,...,706.0087,56.7232,666.0000,676.0000,698.0000,714.0000,718.0000,718.0000,718.0000,0.0012
5,bairong_v1,numerical,12448,0,0.0000,7523,872.2600,24,0.1900,0,...,758.3480,91.5451,537.8692,594.2805,695.4025,767.1300,826.9325,894.5710,930.2000,0.0015
6,xiaoniu_c3,numerical,12448,0,0.0000,514,850,156,1.2500,0,...,610.4062,106.1409,395.0000,455.0000,547.0000,607.0000,673.2500,784.0000,850.0000,0.0015
7,dxm_v6pro,numerical,12448,10184,81.8100,731,46.5500,25,1.1000,0,...,49.0620,4.9509,37.9400,41.2700,45.4575,49.0600,52.6200,57.3100,60.9496,0.0019
8,bcf_fpv1,numerical,12448,10184,81.8100,307,254.0000,37,1.6300,0,...,246.9594,76.3986,55.0000,102.0000,202.0000,257.0000,302.0000,357.0000,384.0000,0.0098
9,apply_for_admission_confidence,numerical,12448,0,0.0000,14,75,3977,31.9500,0,...,75.2032,1.3076,72.0000,73.0000,74.0000,75.0000,76.0000,77.0000,79.0000,0.0001


In [ ]:
### 3.4 数值特征摘要
print("\n" + "="*60)
print("3.4 数值特征摘要 (numeric_summary)")
print("="*60)

numeric_df = eda.numeric_summary(df)
numeric_df.head(20)


3.4 数值特征摘要 (numeric_summary)


,特征名,样本数,均值,标准差,最小值,最大值,中位数,偏度,峰度,1%,5%,95%,99%,零值数,零值率(%),负值数,负值率(%)
0,MOB1,12448,4.0794,15.4973,0.0000,106.0000,0.0000,3.9286,14.6386,0.0000,0.0000,48.0000,77.0000,10838,87.0700,0,0.0000
1,MOB2,12448,6.7825,17.6641,0.0000,106.0000,0.0000,2.7791,7.0790,0.0000,0.0000,52.0000,77.0000,9341,75.0400,0,0.0000
2,bj_qy24,12448,598.5537,67.1235,-999.0000,850.0000,599.0000,-0.9296,25.3889,461.0000,489.0000,707.0000,756.5300,0,0.0000,1,0.0100
3,mobtech80,12448,706.0087,56.7232,-999.0000,718.0000,714.0000,-28.3838,850.0169,666.0000,676.0000,718.0000,718.0000,0,0.0000,13,0.1000
4,bairong_v1,12448,758.3480,91.5451,474.0900,990.3400,767.1300,-0.3439,-0.4005,537.8692,594.2805,894.5710,930.2000,0,0.0000,0,0.0000
5,xiaoniu_c3,12448,610.4062,106.1409,-999.0000,850.0000,607.0000,-2.3841,37.2825,395.0000,455.0000,784.0000,850.0000,0,0.0000,9,0.0700
6,dxm_v6pro,2264,49.0620,4.9509,30.8000,64.1900,49.0600,0.0617,-0.0864,37.9400,41.2700,57.3100,60.9496,0,0.0000,0,0.0000
7,bcf_fpv1,2264,246.9594,76.3986,1.0000,449.0000,257.0000,-0.5708,0.0325,55.0000,102.0000,357.0000,384.0000,0,0.0000,0,0.0000
8,apply_for_admission_confidence,12448,75.2032,1.3076,68.0000,82.0000,75.0000,0.1647,1.0149,72.0000,73.0000,77.0000,79.0000,0,0.0000,0,0.0000
9,apply_for_admission_score,12448,611.1431,33.7297,503.0000,686.0000,622.0000,-1.6662,1.5020,525.0000,526.0000,641.0000,666.0000,0,0.0000,0,0.0000


In [ ]:
### 3.5 类别特征摘要
print("\n" + "="*60)
print("3.5 类别特征摘要 (category_summary)")
print("="*60)

# 包括客户等级字段
cat_df = eda.category_summary(df)
cat_df.head(20)


3.5 类别特征摘要 (category_summary)


,特征名,样本数,类别数,最常见类别,最常见占比(%),类别1,类别1占比(%),类别2,类别2占比(%),类别3,类别3占比(%),类别4,类别4占比(%),类别5,类别5占比(%)
0,customer_grade,12448,5,B,34.1400,B,34.1400,A,21.4000,D,18.0300,E,17.5000,C,8.9300


In [ ]:
### 3.6 数据质量报告
print("\n" + "="*60)
print("3.6 数据质量报告 (data_quality_report)")
print("="*60)

quality_df = eda.data_quality_report(df)
quality_df


3.6 数据质量报告 (data_quality_report)


,特征名,问题类型,严重程度,问题值,建议处理
0,dxm_v6pro,高缺失率,高,81.8%,考虑删除或业务填充
1,bcf_fpv1,高缺失率,高,81.8%,考虑删除或业务填充
2,overdue_amt_sum_6m,准常数特征,中,95.9%,检查业务意义，考虑删除
3,overdue_loan_m0_count_6m,准常数特征,中,95.9%,检查业务意义，考虑删除
4,overdue_loan_m1_count_12m,准常数特征,中,100.0%,检查业务意义，考虑删除
5,overdue_loan_m1_count_24m,准常数特征,中,99.8%,检查业务意义，考虑删除
6,overdue_loan_m1_count_6m,准常数特征,中,100.0%,检查业务意义，考虑删除


## 4. 目标变量分析

需要先确定目标变量（逾期相关字段）

In [ ]:
# 识别目标变量
target_candidates = [col for col in df.columns if any(k in col.lower() for k in ['target', 'fpd', 'dpd', 'sfpd', 'mob1', 'mob2'])]
print(f"候选目标变量: {target_candidates}")

# 选择目标变量
if target_candidates:
    target_col = target_candidates[-1]
    print(f"\n使用目标变量: {target_col}")
    print(f"目标分布:\n{df[target_col].value_counts()}")
else:
    # 创建模拟目标变量
    np.random.seed(42)
    df['target'] = np.random.choice([0, 1], size=len(df), p=[0.85, 0.15])
    target_col = 'target'
    print(f"创建模拟目标变量: {target_col}")

候选目标变量: ['MOB1', 'MOB2', 'FPD15', 'SFPD15']

使用目标变量: SFPD15
目标分布:
SFPD15
0    10724
1     1724
Name: count, dtype: int64


In [ ]:
### 4.1 目标变量分布
print("\n" + "="*60)
print("4.1 目标变量分布 (target_distribution)")
print("="*60)

target_dist = eda.target_distribution(df, target_col=target_col)
target_dist


4.1 目标变量分布 (target_distribution)


,类别,样本数,占比(%),累计占比(%)
0,0,10724,86.1500,86.1500
1,1,1724,13.8500,100.0000


In [ ]:
### 4.2 整体逾期率
print("\n" + "="*60)
print("4.2 整体逾期率 (bad_rate_overall)")
print("="*60)

# 如果有多个月份的数据
date_candidates = [col for col in df.columns if any(k in col.lower() for k in ['date', 'month', 'time', 'period'])]
if date_candidates:
    date_col = date_candidates[0]
    print(f"使用日期字段: {date_col}")
    
    # 确保日期格式正确
    if df[date_col].dtype == 'object':
        try:
            df[date_col] = pd.to_datetime(df[date_col])
        except:
            pass
    
    bad_rate_df = eda.bad_rate_overall(df, target_col=target_col)
else:
    bad_rate_df = eda.bad_rate_overall(df, target_col=target_col)

bad_rate_df


4.2 整体逾期率 (bad_rate_overall)
使用日期字段: loan_date


{'样本总数': 12448, '好样本数': 10724, '坏样本数': 1724, '逾期率(%)': 13.85}

In [ ]:
### 4.3 分维度逾期率（使用客户等级作为维度）
print("\n" + "="*60)
print("4.3 分维度逾期率 (bad_rate_by_dimension)")
print("="*60)

if grade_cols:
    dim_col= grade_cols[0]
    print(f"分析维度: {dim_col}")
    
    bad_rate_dim = eda.bad_rate_by_dimension(
        df, 
        target_col=target_col,
        dim_col=dim_col
    )

bad_rate_dim


4.3 分维度逾期率 (bad_rate_by_dimension)
分析维度: customer_grade


,维度值,样本数,坏样本数,好样本数,逾期率(%),样本占比(%)
0,B,4250,705,3545,16.5900,34.1400
1,A,2664,385,2279,14.4500,21.4000
2,C,1112,157,955,14.1200,8.9300
3,E,2178,261,1917,11.9800,17.5000
4,D,2244,216,2028,9.6300,18.0300


In [ ]:
### 4.4 逾期率趋势
print("\n" + "="*60)
print("4.4 逾期率趋势 (bad_rate_trend)")
print("="*60)

if date_candidates:
    trend_df = eda.bad_rate_trend(
        df,
        target_col=target_col,
        date_col=date_col,
        freq='M'
    )

trend_df.head(20)


4.4 逾期率趋势 (bad_rate_trend)


,时间周期,样本数,好样本数,坏样本数,逾期率(%),环比变化(%)
0,2024-11,1,1,0,0.0000,NaN
1,2024-12,14,14,0,0.0000,0.0000
2,2025-01,231,231,0,0.0000,0.0000
3,2025-02,105,105,0,0.0000,0.0000
4,2025-03,519,519,0,0.0000,0.0000
5,2025-04,597,597,0,0.0000,0.0000
6,2025-05,467,467,0,0.0000,0.0000
7,2025-06,469,459,10,2.1300,2.1300
8,2025-07,2903,2404,499,17.1900,15.0600
9,2025-08,7142,5927,1215,17.0100,-0.1800


In [ ]:
### 4.5 分箱逾期率
print("\n" + "="*60)
print("4.5 分箱逾期率 (bad_rate_by_bins)")
print("="*60)

# 选择数值特征进行分箱
if score_cols:
    feature_col = score_cols[0]
    print(f"分析特征: {feature_col}")
    
    bad_rate_bins = eda.bad_rate_by_bins(
        df,
        score_col=feature_col,
        target_col=target_col,
        n_bins=5,
        method='quantile'
    )

bad_rate_bins


4.5 分箱逾期率 (bad_rate_by_bins)
分析特征: apply_for_admission_score


,分箱,分箱区间,样本数,好样本数,坏样本数,逾期率(%),样本占比(%),提升度
0,1,"(502.999, 618.0]",3446,3071,375,10.8800,27.6800,0.7857
1,2,"(618.0, 621.0]",2088,1829,259,12.4000,16.7700,0.8956
2,3,"(621.0, 622.0]",2587,2136,451,17.4300,20.7800,1.2588
3,4,"(622.0, 628.0]",1841,1564,277,15.0500,14.7900,1.0864
4,5,"(628.0, 686.0]",2486,2124,362,14.5600,19.9700,1.0514


In [ ]:
### 4.6 样本分布
print("\n" + "="*60)
print("4.6 样本分布 (sample_distribution)")
print("="*60)

if date_candidates:
    sample_dist = eda.sample_distribution(df, date_col=date_col, freq='M')

sample_dist.head(20)


4.6 样本分布 (sample_distribution)


,时间周期,样本数,样本环比(%)
0,2024-11,1,NaN
1,2024-12,14,1300.0000
2,2025-01,231,1550.0000
3,2025-02,105,-54.5500
4,2025-03,519,394.2900
5,2025-04,597,15.0300
6,2025-05,467,-21.7800
7,2025-06,469,0.4300
8,2025-07,2903,518.9800
9,2025-08,7142,146.0200


## 5. 特征分析

In [ ]:
### 5.1 特征类型推断
print("\n" + "="*60)
print("5.1 特征类型推断 (feature_type_inference)")
print("="*60)

type_inference = eda.feature_type_inference(df)
# print(f"数值特征数: {len(type_inference['number'])}")
# print(f"类别特征数: {len(type_inference['categorical'])}")
# print(f"\n数值特征: {type_inference['number'][:10]}")
# print(f"\n类别特征: {type_inference['categorical'][:10]}")
type_inference


5.1 特征类型推断 (feature_type_inference)


,特征名,特征类型,数据类型,唯一值数,建议处理方式
0,MOB1,numerical,int64,95,标准化/归一化
1,MOB2,numerical,int64,105,标准化/归一化
2,loan_date,datetime,datetime64[ns],223,提取时间特征
3,bj_qy24,numerical,int64,349,标准化/归一化
4,mobtech80,numerical,int64,22,标准化/归一化
...,...,...,...,...,...
81,performance_loan_sum_3m,numerical,int64,11,标准化/归一化
82,performance_loan_sum_6m,numerical,int64,11,标准化/归一化
83,FPD15,numerical,int64,2,标准化/归一化
84,SFPD15,numerical,int64,2,标准化/归一化


In [ ]:
### 5.2 数值特征分布
print("\n" + "="*60)
print("5.2 数值特征分布 (numeric_distribution)")
print("="*60)

if score_cols:
    num_dist = eda.numeric_distribution(df, feature=score_cols[0])

num_dist


5.2 数值特征分布 (numeric_distribution)


,分箱编号,分箱区间,频数,频率(%),累计频率(%)
0,1,"[503.00, 512.15)",6,0.0500,0.0500
1,2,"[512.15, 521.30)",10,0.0800,0.1300
2,3,"[521.30, 530.45)",904,7.2600,7.3900
3,4,"[530.45, 539.60)",227,1.8200,9.2100
4,5,"[539.60, 548.75)",491,3.9400,13.1600
5,6,"[548.75, 557.90)",317,2.5500,15.7100
6,7,"[557.90, 567.05)",0,0.0000,15.7100
7,8,"[567.05, 576.20)",2,0.0200,15.7200
8,9,"[576.20, 585.35)",18,0.1400,15.8700
9,10,"[585.35, 594.50)",4,0.0300,15.9000


In [ ]:
### 5.3 类别特征分布
print("\n" + "="*60)
print("5.3 类别特征分布 (categorical_distribution)")
print("="*60)

if grade_cols:
    cat_dist = eda.categorical_distribution(df, feature=grade_cols[0])

cat_dist.head(30)


5.3 类别特征分布 (categorical_distribution)


,类别值,频数,频率(%),累计频率(%)
0,B,4250,34.1400,34.1400
1,A,2664,21.4000,55.5400
2,D,2244,18.0300,73.5700
3,E,2178,17.5000,91.0700
4,C,1112,8.9300,100.0000


In [ ]:
### 5.4 异常值检测
print("\n" + "="*60)
print("5.4 异常值检测 (outlier_detection)")
print("="*60)

if score_cols:
    outlier_df = eda.outlier_detection(df, features=score_cols[:5], method='iqr')

outlier_df.head(20)


5.4 异常值检测 (outlier_detection)


,特征名,异常值数,异常值率(%),正常范围,最小值,最大值
0,apply_for_admission_score,3533,28.3800,"[609.00, 633.00]",503,686
1,loan_behavior_score,2302,18.4900,"[592.00, 632.00]",479,680


In [ ]:
### 5.5 罕见类别检测
print("\n" + "="*60)
print("5.5 罕见类别检测 (rare_category_detection)")
print("="*60)

if grade_cols:
    rare_cat = eda.rare_category_detection(df, features=grade_cols, threshold=0.01)

rare_cat


5.5 罕见类别检测 (rare_category_detection)


,信息
0,未发现稀有类别


In [ ]:
### 5.6 集中度分析
print("\n" + "="*60)
print("5.6 集中度分析 (concentration_analysis)")
print("="*60)

if grade_cols:
    concentration = eda.concentration_analysis(df, features=grade_cols)

concentration


5.6 集中度分析 (concentration_analysis)


,特征名,Gini系数,集中度评级


In [ ]:
### 3.4 数值特征摘要
print("\n" + "="*60)
print("3.4 数值特征摘要 (numeric_summary)")
print("="*60)

numeric_df = eda.numeric_summary(df)
numeric_df.head(20)


3.4 数值特征摘要 (numeric_summary)


,特征名,样本数,均值,标准差,最小值,最大值,中位数,偏度,峰度,1%,5%,95%,99%,零值数,零值率(%),负值数,负值率(%)
0,MOB1,12448,4.0794,15.4973,0.0000,106.0000,0.0000,3.9286,14.6386,0.0000,0.0000,48.0000,77.0000,10838,87.0700,0,0.0000
1,MOB2,12448,6.7825,17.6641,0.0000,106.0000,0.0000,2.7791,7.0790,0.0000,0.0000,52.0000,77.0000,9341,75.0400,0,0.0000
2,bj_qy24,12448,598.5537,67.1235,-999.0000,850.0000,599.0000,-0.9296,25.3889,461.0000,489.0000,707.0000,756.5300,0,0.0000,1,0.0100
3,mobtech80,12448,706.0087,56.7232,-999.0000,718.0000,714.0000,-28.3838,850.0169,666.0000,676.0000,718.0000,718.0000,0,0.0000,13,0.1000
4,bairong_v1,12448,758.3480,91.5451,474.0900,990.3400,767.1300,-0.3439,-0.4005,537.8692,594.2805,894.5710,930.2000,0,0.0000,0,0.0000
5,xiaoniu_c3,12448,610.4062,106.1409,-999.0000,850.0000,607.0000,-2.3841,37.2825,395.0000,455.0000,784.0000,850.0000,0,0.0000,9,0.0700
6,dxm_v6pro,2264,49.0620,4.9509,30.8000,64.1900,49.0600,0.0617,-0.0864,37.9400,41.2700,57.3100,60.9496,0,0.0000,0,0.0000
7,bcf_fpv1,2264,246.9594,76.3986,1.0000,449.0000,257.0000,-0.5708,0.0325,55.0000,102.0000,357.0000,384.0000,0,0.0000,0,0.0000
8,apply_for_admission_confidence,12448,75.2032,1.3076,68.0000,82.0000,75.0000,0.1647,1.0149,72.0000,73.0000,77.0000,79.0000,0,0.0000,0,0.0000
9,apply_for_admission_score,12448,611.1431,33.7297,503.0000,686.0000,622.0000,-1.6662,1.5020,525.0000,526.0000,641.0000,666.0000,0,0.0000,0,0.0000


In [ ]:
### 5.7 特征时间稳定性
print("\n" + "="*60)
print("5.7 特征时间稳定性 (feature_stability_over_time)")
print("="*60)

if date_candidates and score_cols:
    stability_df = eda.feature_stability_over_time(
        df,
        features=score_cols[:3],
        date_col=date_col,
        freq='M'
    )

stability_df.head(30)


5.7 特征时间稳定性 (feature_stability_over_time)


,特征名,均值标准差,均值变异系数,稳定性评级,统计期数
0,apply_for_admission_score,6.0713,0.0100,非常稳定,10
1,loan_behavior_score,4.4488,0.0074,非常稳定,10


## 6. 特征与标签关系分析

In [ ]:
### 6.1 IV分析
print("\n" + "="*60)
print("6.1 IV分析 (iv_analysis)")
print("="*60)

if score_cols:
    feature_col = score_cols[0]
    iv_result = eda.iv_analysis(
        df,
        feature=feature_col,
        target=target_col,
        n_bins=5
    )
    print(f"\n特征 '{feature_col}' 的 IV 值:")
    print(iv_result)


6.1 IV分析 (iv_analysis)

特征 'apply_for_admission_score' 的 IV 值:
{'特征名': 'apply_for_admission_score', 'IV值': 0.0481, '预测能力': '弱预测能力', '分箱数': 5, '分箱明细':    分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 618.0]  2178  1917   261 0.1750 0.1788 0.1514 0.1198 -0.1662 0.0045 0.0481 0.8653 -0.0286   0.8653 -0.0286    1917     261 0.0274
1   1  (618.0, 621.0]  2244  2028   216 0.1803 0.1891 0.1253 0.0963 -0.4117 0.0263 0.0481 0.6950 -0.0671   0.7789 -0.1218    3945     477 0.0912
2   2  (621.0, 622.0]  1112   955   157 0.0893 0.0891 0.0911 0.1412  0.0224 0.0000 0.0481 1.0194  0.0019   0.8272 -0.1383    4900     634 0.0892
3   3  (622.0, 628.0]  4250  3545   705 0.3414 0.3306 0.4089 0.1659  0.2127 0.0167 0.0481 1.1977  0.1025   0.9882 -0.0435    8445    1339 0.0108
4   4   (628.0, +inf]  2664  2279   385 0.2140 0.2125 0.2233 0.1445  0.0496 0.0005 0.0481 1.0435  0.0118   1.0000  0.0000   

In [ ]:
### 6.2 批量IV分析
print("\n" + "="*60)
print("6.2 批量IV分析 (batch_iv_analysis)")
print("="*60)

# 选择数值特征进行批量IV分析
if score_cols and len(score_cols) >= 2:
    batch_iv = eda.batch_iv_analysis(
        df,
        features=score_cols[:10],
        target=target_col,
        n_bins=5
    )

batch_iv.head(20)


6.2 批量IV分析 (batch_iv_analysis)


,特征名,IV值,预测能力,分箱数
0,apply_for_admission_score,0.0481,弱预测能力,5
1,loan_behavior_score,0.0430,弱预测能力,5


In [ ]:
### 6.3 WOE分析
print("\n" + "="*60)
print("6.3 WOE分析 (woe_analysis)")
print("="*60)

if score_cols:
    feature_col = score_cols[0]
    woe_result = eda.woe_analysis(
        df,
        feature=feature_col,
        target=target_col,
        n_bins=5
    )
    woe_result


6.3 WOE分析 (woe_analysis)


In [ ]:
### 6.4 分箱逾期率
print("\n" + "="*60)
print("6.4 分箱逾期率 (binning_bad_rate)")
print("="*60)

if score_cols:
    feature_col = score_cols[0]
    bin_bad_rate = eda.binning_bad_rate(
        df,
        feature=feature_col,
        target=target_col,
        n_bins=5,
        method='quantile'
    )

bin_bad_rate


6.4 分箱逾期率 (binning_bad_rate)


,分箱,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,0,"(-inf, 618.0]",2178,1917,261,0.1750,0.1788,0.1514,0.1198,-0.1662,0.0045,0.0481,0.8653,-0.0286,0.8653,-0.0286,1917,261,0.0274
1,1,"(618.0, 621.0]",2244,2028,216,0.1803,0.1891,0.1253,0.0963,-0.4117,0.0263,0.0481,0.6950,-0.0671,0.7789,-0.1218,3945,477,0.0912
2,2,"(621.0, 622.0]",1112,955,157,0.0893,0.0891,0.0911,0.1412,0.0224,0.0000,0.0481,1.0194,0.0019,0.8272,-0.1383,4900,634,0.0892
3,3,"(622.0, 628.0]",4250,3545,705,0.3414,0.3306,0.4089,0.1659,0.2127,0.0167,0.0481,1.1977,0.1025,0.9882,-0.0435,8445,1339,0.0108
4,4,"(628.0, +inf]",2664,2279,385,0.2140,0.2125,0.2233,0.1445,0.0496,0.0005,0.0481,1.0435,0.0118,1.0000,0.0000,10724,1724,0.0000


In [ ]:
### 6.5 单调性检查
print("\n" + "="*60)
print("6.5 单调性检查 (monotonicity_check)")
print("="*60)

if score_cols:
    mono_results = []
    for feat in score_cols[:5]:
        result = eda.monotonicity_check(
            df,
            feature=feat,
            target=target_col,
            n_bins=5
        )
        mono_results.append(result)
    mono_result = pd.DataFrame(mono_results)
mono_result



6.5 单调性检查 (monotonicity_check)


,特征名,单调性,单调方向,Spearman相关系数,P值,说明
0,apply_for_admission_score,单调,递增,0.8000,0.1041,需检查业务含义
1,loan_behavior_score,单调,递减,-0.7000,0.1881,需检查业务含义


In [ ]:
### 6.6 单变量AUC
print("\n" + "="*60)
print("6.6 单变量AUC (univariate_auc)")
print("="*60)

if score_cols:
    auc_results = []
    for feat in score_cols[:10]:
        result = eda.univariate_auc(
            df,
            feature=feat,
            target=target_col
        )
        auc_results.append(result)
    auc_result = pd.DataFrame(auc_results)
auc_result.head(20)



6.6 单变量AUC (univariate_auc)


,特征名,AUC值,区分能力
0,apply_for_admission_score,0.5365,弱区分能力
1,loan_behavior_score,0.4696,无区分能力(倒置)


In [ ]:
### 6.7 特征重要性排序
print("\n" + "="*60)
print("6.7 特征重要性排序 (feature_importance_ranking)")
print("="*60)

if score_cols:
    importance = eda.feature_importance_ranking(
        df,
        features=score_cols[:10],
        target=target_col,
        metrics='iv'
    )
importance.head(20)


6.7 特征重要性排序 (feature_importance_ranking)


,特征名,IV值,预测能力,IV得分,综合得分,排名
0,apply_for_admission_score,0.0481,弱预测能力,1.0000,1.0000,1
1,loan_behavior_score,0.0430,弱预测能力,0.8940,0.8940,2


## 7. 相关性分析

In [ ]:
### 7.1 相关系数矩阵
print("\n" + "="*60)
print("7.1 相关系数矩阵 (correlation_matrix)")
print("="*60)

if score_cols and len(score_cols) >= 2:
    corr_matrix = eda.correlation_matrix(
        df,
        features=score_cols[:10],
        method='pearson'
    )
corr_matrix


7.1 相关系数矩阵 (correlation_matrix)


,apply_for_admission_score,loan_behavior_score
apply_for_admission_score,1.0000,0.9679
loan_behavior_score,0.9679,1.0000


In [ ]:
### 7.2 高相关特征对
print("\n" + "="*60)
print("7.2 高相关特征对 (high_correlation_pairs)")
print("="*60)

if score_cols and len(score_cols) >= 2:
    high_corr = eda.high_correlation_pairs(
        df,
        features=score_cols[:15],
        threshold=0.7
    )
high_corr.head(20)


7.2 高相关特征对 (high_correlation_pairs)


,特征1,特征2,相关系数,绝对相关系数,相关评级
0,apply_for_admission_score,loan_behavior_score,0.9679,0.9679,无共线性


In [ ]:
### 7.3 相关性过滤
print("\n" + "="*60)
print("7.3 相关性过滤 (correlation_filter)")
print("="*60)

if score_cols and len(score_cols) >= 2:
    corr_filter = eda.correlation_filter(
        df,
        features=score_cols[:15],
        target=target_col,
        threshold=0.8
    )
    print(f"保留特征数: {len(corr_filter)}")
    print(f"剔除特征: {corr_filter}")


7.3 相关性过滤 (correlation_filter)
保留特征数: 1
剔除特征: ['apply_for_admission_score']


In [ ]:
### 7.4 VIF分析
print("\n" + "="*60)
print("7.4 VIF分析 (vif_analysis)")
print("="*60)

if score_cols:
    vif_result = eda.vif_analysis(df, features=score_cols[:10])
vif_result.head(20)


7.4 VIF分析 (vif_analysis)


,特征名,VIF值,共线性评级,建议
0,apply_for_admission_score,4787.8600,严重共线性,剔除
1,loan_behavior_score,4787.8600,严重共线性,剔除


## 8. 稳定性分析

In [ ]:
### 8.1 PSI分析
print("\n" + "="*60)
print("8.1 PSI分析 (psi_analysis)")
print("="*60)

# 将数据分为基准集和测试集（按时间）
if date_candidates:
    # 按时间划分
    median_date = df[date_col].median()
    base_df = df[df[date_col] <= median_date]
    test_df = df[df[date_col] > median_date]
    
    print(f"基准集样本数: {len(base_df)}")
    print(f"测试集样本数: {len(test_df)}")
    
    if score_cols and len(base_df) > 10 and len(test_df) > 10:
        feature_col = score_cols[0]
        psi_result = eda.psi_analysis(base_df, test_df, feature_col,
            n_bins=10
        )
        print(f"\n特征 '{feature_col}' 的 PSI: {psi_result['psi']:.4f}")
psi_result['table']


8.1 PSI分析 (psi_analysis)
基准集样本数: 6229
测试集样本数: 6219


KeyError: '分档PSI值'

In [ ]:
### 8.2 批量PSI分析
print("\n" + "="*60)
print("8.2 批量PSI分析 (batch_psi_analysis)")
print("="*60)

if date_candidates and score_cols and len(base_df) > 10 and len(test_df) > 10:
    batch_psi = eda.batch_psi_analysis(
        base_df,
        test_df,
        features=score_cols[:10],
        n_bins=10
    )
batch_psi.head(20)


8.2 批量PSI分析 (batch_psi_analysis)


TypeError: batch_psi_analysis() got multiple values for argument 'features'

In [ ]:
### 8.3 CSI分析
print("\n" + "="*60)
print("8.3 CSI分析 (csi_analysis)")
print("="*60)

if date_candidates and score_cols and len(base_df) > 10 and len(test_df) > 10:
    feature= score_cols[0]
    csi_result = eda.csi_analysis(
        base_df,
        test_df,
        feature=feature_col,
        target=target_col,
        n_bins=5
    )
csi_result.head(20)

In [ ]:
### 8.4 时间PSI追踪
print("\n" + "="*60)
print("8.4 时间PSI追踪 (time_psi_tracking)")
print("="*60)

if date_candidates and score_cols:
    time_psi = eda.time_psi_tracking(
        df,
        features=score_cols[:3],
        date_col=date_col,
        freq='M',
        n_bins=10
    )
time_psi.head(30)

In [ ]:
### 8.5 稳定性报告
print("\n" + "="*60)
print("8.5 稳定性报告 (stability_report)")
print("="*60)

if date_candidates and score_cols:
    stability_rpt = eda.stability_report(
        df,
        features=score_cols[:10],
        date_col=date_col,
        psi_threshold=0.1
    )
stability_rpt.head(20)

In [ ]:
### 8.6 PSI交叉分析
print("\n" + "="*60)
print("8.6 PSI交叉分析 (psi_cross_analysis)")
print("="*60)

if date_candidates and score_cols:
    # 按维度交叉分析
    if grade_cols:
        psi_cross = eda.psi_cross_analysis(
            df,
            features=score_cols[:5],
            date_col=date_col,
            freq='M'
        )
psi_cross.head(30)

## 9. Vintage分析

In [ ]:
### 9.1 Vintage分析
print("\n" + "="*60)
print("9.1 Vintage分析 (vintage_analysis)")
print("="*60)

# 查找MOB和放款日期字段
mob_candidates = [col for col in df.columns if 'mob' in col.lower()]
issue_candidates = [col for col in df.columns if any(k in col.lower() for k in ['issue', 'loan', 'grant', 'originate'])]

print(f"MOB字段: {mob_candidates}")
print(f"放款日期字段: {issue_candidates}")

if mob_candidates and target_col:
    mob_col = mob_candidates[0]
    
    # 检查MOB字段是否数值型
    if pd.api.types.is_numeric_dtype(df[mob_col]):
        vintage_result = eda.vintage_analysis(df, vintage_col=issue_candidates[0] if issue_candidates else None
        , mob_col=mob_col, target_col=target_col)
vintage_result.head(30)

In [ ]:
### 9.2 Vintage汇总
print("\n" + "="*60)
print("9.2 Vintage汇总 (vintage_summary)")
print("="*60)

if mob_candidates and issue_candidates:
    vintage_sum = eda.vintage_summary(
        df,
        mob_col=mob_col,
        target_col=target_col,
        vintage_col=issue_candidates[0]
    )
vintage_sum.head(30)

In [ ]:
### 9.3 滚动率分析
print("\n" + "="*60)
print("9.3 滚动率分析 (roll_rate_analysis)")
print("="*60)

# 需要多个逾期天数字段
dpd_candidates = [col for col in df.columns if 'dpd' in col.lower()]
print(f"逾期字段: {dpd_candidates}")

if len(dpd_candidates) >= 2:
    roll_rate = eda.roll_rate_analysis(
        df,
        overdue_cols=dpd_candidates[:3]
    )
roll_rate

## 10. 特征分组分析

In [ ]:
### 10.1 特征分组分析
print("\n" + "="*60)
print("10.1 特征分组分析 (feature_group_analysis)")
print("="*60)

# 按客户等级分组分析特征
if grade_cols and score_cols:
    group_analysis = eda.feature_group_analysis(
        df,
        features=score_cols[:5],
        group_cols=grade_cols[0],
        target_col=target_col
    )
group_analysis.head(30)

In [ ]:
### 10.2 群体稳定性监控
print("\n" + "="*60)
print("10.2 群体稳定性监控 (population_stability_monitor)")
print("="*60)

if date_candidates and grade_cols:
    psm_result = eda.population_stability_monitor(
        df,
        segment_cols=grade_cols[0],
        date_col=date_col,
        freq='M',
        metrics=['count', 'bad_rate', 'iv']
    )
psm_result.head(30)

## 11. 综合报告

In [ ]:
### 11.1 EDA摘要
print("\n" + "="*60)
print("11.1 EDA摘要 (eda_summary)")
print("="*60)

eda_sum = eda.eda_summary(df, target=target_col if target_col else None)

# 展示各个部分
for key, value in eda_sum.items():
    print(f"\n{'='*60}")
    print(f"{key}")
    print(f"{'='*60}")
    if isinstance(value, pd.DataFrame):
        display(value.head(10))
    else:
        print(value)

In [ ]:
### 11.2 生成完整报告
print("\n" + "="*60)
print("11.2 生成完整报告 (generate_report)")
print("="*60)

report = eda.generate_report(
    df,
    target=target_col if target_col else None,
    features=score_cols[:20] if score_cols else None,
    include_charts=False
)

print(f"报告包含 {len(report)} 个部分:")
for key in report.keys():
    print(f"  - {key}")

In [ ]:
### 11.3 导出报告到Excel
print("\n" + "="*60)
print("11.3 导出报告到Excel (export_report_to_excel)")
print("="*60)

# 导出报告
output_path = 'eda_report.xlsx'
eda.export_report_to_excel(report, output_path)
print(f"报告已导出到: {output_path}")

## 12. 可视化演示

结合 hscredit.core.viz 模块进行可视化

In [ ]:
from hscredit.core.viz import bin_plot, hist_plot, corr_plot, ks_plot, psi_plot

### 12.1 分箱图
print("\n" + "="*60)
print("12.1 分箱图 (bin_plot)")
print("="*60)

if score_cols:
    # 获取分箱统计表
    bin_stats = eda.binning_bad_rate(
        df,
        feature=score_cols[0],
        target=target_col,
        n_bins=5
    )
    
    # 绘制分箱图
    fig = bin_plot(bin_stats, title=f'{score_cols[0]} 分箱图')
    plt.show()

In [ ]:
### 12.2 分布图
print("\n" + "="*60)
print("12.2 分布图 (hist_plot)")
print("="*60)

if score_cols:
    fig = hist_plot(df, feature=score_cols[0], target=target_col, bins=20)
    plt.show()

In [ ]:
### 12.3 相关性热力图
print("\n" + "="*60)
print("12.3 相关性热力图 (corr_plot)")
print("="*60)

if score_cols and len(score_cols) >= 2:
    fig = corr_plot(df, features=score_cols[:10], method='pearson', figsize=(12, 10))
    plt.show()

## 13. 总结

本示例演示了 hscredit.core.eda 模块的完整功能：

1. **数据概览与质量评估** (8个函数)
   - `data_info`: 数据基本信息
   - `missing_analysis`: 缺失值分析
   - `feature_summary`: 特征摘要
   - `numeric_summary`: 数值特征摘要
   - `category_summary`: 类别特征摘要
   - `data_quality_report`: 数据质量报告
   - `feature_group_analysis`: 特征分组分析
   - `population_stability_monitor`: 群体稳定性监控

2. **目标变量分析** (6个函数)
   - `target_distribution`: 目标分布
   - `bad_rate_overall`: 整体逾期率
   - `bad_rate_by_dimension`: 分维度逾期率
   - `bad_rate_trend`: 逾期率趋势
   - `bad_rate_by_bins`: 分箱逾期率
   - `sample_distribution`: 样本分布

3. **特征分析** (7个函数)
   - `feature_type_inference`: 特征类型推断
   - `numeric_distribution`: 数值分布
   - `categorical_distribution`: 类别分布
   - `outlier_detection`: 异常值检测
   - `rare_category_detection`: 罕见类别检测
   - `concentration_analysis`: 集中度分析
   - `feature_stability_over_time`: 特征时间稳定性

4. **特征标签关系** (7个函数)
   - `iv_analysis`: IV分析
   - `batch_iv_analysis`: 批量IV分析
   - `woe_analysis`: WOE分析
   - `binning_bad_rate`: 分箱逾期率
   - `monotonicity_check`: 单调性检查
   - `univariate_auc`: 单变量AUC
   - `feature_importance_ranking`: 特征重要性排序

5. **相关性分析** (4个函数)
   - `correlation_matrix`: 相关系数矩阵
   - `high_correlation_pairs`: 高相关特征对
   - `correlation_filter`: 相关性过滤
   - `vif_analysis`: VIF分析

6. **稳定性分析** (6个函数)
   - `psi_analysis`: PSI分析
   - `batch_psi_analysis`: 批量PSI分析
   - `csi_analysis`: CSI分析
   - `time_psi_tracking`: 时间PSI追踪
   - `stability_report`: 稳定性报告
   - `psi_cross_analysis`: PSI交叉分析

7. **Vintage分析** (3个函数)
   - `vintage_analysis`: Vintage分析
   - `vintage_summary`: Vintage汇总
   - `roll_rate_analysis`: 滚动率分析

8. **综合报告** (3个函数)
   - `eda_summary`: EDA摘要
   - `generate_report`: 生成完整报告
   - `export_report_to_excel`: 导出Excel报告

**特色功能**：
- 客户等级自动分箱（A~E + N缺失）
- 金融场景专用指标（IV、PSI、CSI、VIF等）
- 中文输出，便于理解和报告
- 一键生成完整数据探索报告